In [ ]:
import pandas as pd

In [9]:
# load the file into a DataFrame
df = pd.read_csv('ravenstack_churn_events.csv')
df.head()

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,NaN
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive


In [11]:
# shape of the data
df.shape

(600, 9)

In [12]:
# columns in the DataFrame
df.columns

Index(['churn_event_id', 'account_id', 'churn_date', 'reason_code',
       'refund_amount_usd', 'preceding_upgrade_flag',
       'preceding_downgrade_flag', 'is_reactivation', 'feedback_text'],
      dtype='object')

In [4]:
# check dataframe information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   churn_event_id            600 non-null    object 
 1   account_id                600 non-null    object 
 2   churn_date                600 non-null    object 
 3   reason_code               600 non-null    object 
 4   refund_amount_usd         600 non-null    float64
 5   preceding_upgrade_flag    600 non-null    bool   
 6   preceding_downgrade_flag  600 non-null    bool   
 7   is_reactivation           600 non-null    bool   
 8   feedback_text             452 non-null    object 
dtypes: bool(3), float64(1), object(5)
memory usage: 30.0+ KB


In [22]:
# check for missing values
df.isnull().sum()

churn_event_id                0
account_id                    0
churn_date                    0
reason_code                   0
refund_amount_usd             0
preceding_upgrade_flag        0
preceding_downgrade_flag      0
is_reactivation               0
feedback_text               148
dtype: int64

In [15]:
# check for duplicates
df.duplicated().sum()

np.int64(0)

In [19]:
# change churn_date column format to data_time format
df["churn_date"] = pd.to_datetime(df["churn_date"], errors="coerce")

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   churn_event_id            600 non-null    object        
 1   account_id                600 non-null    object        
 2   churn_date                600 non-null    datetime64[ns]
 3   reason_code               600 non-null    object        
 4   refund_amount_usd         600 non-null    float64       
 5   preceding_upgrade_flag    600 non-null    bool          
 6   preceding_downgrade_flag  600 non-null    bool          
 7   is_reactivation           600 non-null    bool          
 8   feedback_text             452 non-null    object        
dtypes: bool(3), datetime64[ns](1), float64(1), object(4)
memory usage: 30.0+ KB


In [24]:
# fill null falues
df["feedback_text"] = df["feedback_text"].fillna("no feedback")

In [26]:
df.isnull().sum()

churn_event_id              0
account_id                  0
churn_date                  0
reason_code                 0
refund_amount_usd           0
preceding_upgrade_flag      0
preceding_downgrade_flag    0
is_reactivation             0
feedback_text               0
dtype: int64

In [31]:
# findout the outliers

q1 = df["refund_amount_usd"].quantile(0.25)
q3 = df["refund_amount_usd"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = df[(df["refund_amount_usd"] < lower) | (df["refund_amount_usd"] > upper)]
print(outliers[["churn_event_id", "refund_amount_usd"]].head())

   churn_event_id  refund_amount_usd
0        C-816288               4.03
1        C-5a81e7              96.45
3        C-accb39              54.94
6        C-2e2966              34.40
10       C-c89493             261.49


In [34]:
# featuring engineering
df["refund_bucket"] = pd.cut(
    df["refund_amount_usd"],
    bins=[-1, 0, 25, 100, float("inf")],
    labels=["zero", "low", "medium", "high"]
)

df["churn_month"] = df["churn_date"].dt.to_period("M").astype(str)
df["had_refund"] = df["refund_amount_usd"] > 0
df

,churn_event_id,account_id,churn_date,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation,feedback_text,refund_bucket,churn_month,had_refund
0,C-816288,A-c37cab,2024-10-27,pricing,4.03,False,False,False,switched to competitor,low,2024-10,True
1,C-5a81e7,A-37f969,2024-06-25,support,96.45,True,False,False,no feedback,medium,2024-06,True
2,C-a174be,A-b07346,2024-11-12,budget,0.00,False,False,False,missing features,zero,2024-11,False
3,C-accb39,A-1e50e0,2023-11-01,budget,54.94,False,False,False,switched to competitor,medium,2023-11,True
4,C-92f889,A-956988,2024-12-30,unknown,0.00,False,True,True,too expensive,zero,2024-12,False
...,...,...,...,...,...,...,...,...,...,...,...,...
595,C-b56a35,A-702032,2024-06-14,competitor,0.00,False,False,False,switched to competitor,zero,2024-06,False
596,C-7e7822,A-dbc825,2024-02-03,budget,0.00,False,False,False,no feedback,zero,2024-02,False
597,C-484fba,A-0a282f,2024-12-31,support,62.66,False,True,False,too expensive,medium,2024-12,True
598,C-22f536,A-e5d6ab,2024-05-11,competitor,0.00,True,False,False,no feedback,zero,2024-05,False


In [36]:
df.isnull().sum()

churn_event_id              0
account_id                  0
churn_date                  0
reason_code                 0
refund_amount_usd           0
preceding_upgrade_flag      0
preceding_downgrade_flag    0
is_reactivation             0
feedback_text               0
refund_bucket               0
churn_month                 0
had_refund                  0
dtype: int64

In [37]:
df.duplicated().sum()

np.int64(0)

In [35]:
# save cleaned file
df.to_csv("ravenstack_churn_events_cleaned.csv", index=False)
print("Saved cleaned dataset")

Saved cleaned dataset
